# Tutorial 01 — Drive the counter via the `gf180-docker-digital` agent (TUI)

> **EXPERIMENTAL.** This tutorial uses the [`eda-agents`](https://github.com/Mauricio-xx/eda-agents) companion package and either Claude Code or opencode to drive LibreLane on your behalf. It is **not part of the chipathon tapeout signoff path**. For tapeout work stay with `examples/`.

This notebook is the **orchestrator**: it stages the same 4-bit counter you saw in `examples/01_rtl2gds_counter.ipynb`, runs the cocotb sanity check, and then prints the **two invocation paths** for the `gf180-docker-digital` agent. You drive the agent in a separate terminal — chat with it, watch it call LibreLane, come back here for the metrics check.

What you will see the agent do:

1. Load the `flow.rtl2gds_gf180_docker` skill via MCP.
2. Confirm the bind-mount path with you.
3. Check the container is up (or start it).
4. Read `rtl/counter.v` and `librelane/config.yaml`.
5. Optionally run `make test-counter` for a quick cocotb sanity.
6. Run LibreLane (full or `nodrc`).
7. Parse `final/metrics.csv` and surface signoff status.

**Pre-requisites** (one-time, see [`../docs/eda_agents_setup.md`](../docs/eda_agents_setup.md)):

- `gf180` container running (run `scripts/bootstrap_container.sh` from repo root if not).
- `eda-agents` pip-installed in an active venv.
- One of: Claude Code CLI **or** opencode CLI on PATH.
- `eda-init` run from this repo root (creates `.mcp.json`, `.opencode.json`, `.claude/agents/`).


## Step 0 — Configuration + path guard

All knobs and the path guard live here. Tweak then run top-to-bottom.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import textwrap

# --- Toggles ---------------------------------------------------------
RUN_PREFLIGHT      = True   # check container, eda-agents, claude/opencode
RUN_STAGE          = True   # copy rtl/ tb/ librelane/ into the workspace
RUN_COCOTB_SANITY  = False  # quick RTL sim, ~10s -- prove the TB works before agent
PRINT_INVOCATIONS  = True   # show the two `claude` / `opencode` commands to copy
RUN_METRICS_CHECK  = False  # parse final/metrics.csv after agent finishes

# --- Path guard (commit 99b3085 pattern) -----------------------------
try:
    NB_DIR = Path(__file__).resolve().parent
except NameError:
    NB_DIR = Path.cwd().resolve()

for _sub in ('rtl', 'tb', 'librelane'):
    if not (NB_DIR / _sub).is_dir():
        raise RuntimeError(
            f'NB_DIR={NB_DIR} does not contain {_sub}/. '
            'Launch this notebook from tutorials/01_counter_with_agent_tui/ '
            'so rtl/ tb/ librelane/ resolve correctly.'
        )

# --- Container + workspace -------------------------------------------
CONTAINER_NAME = os.environ.get('GF180_CONTAINER', 'gf180')
HOST_WORKSPACE = Path.home() / 'eda' / 'designs' / 'eda_agents_counter_tui'
CONTAINER_PROJ = '/foss/designs/eda_agents_counter_tui'

# --- Repo paths ------------------------------------------------------
REPO_ROOT = NB_DIR.parents[1]   # chipathon-gf180mcu-librelane-examples/

print(f'NB_DIR         : {NB_DIR}')
print(f'REPO_ROOT      : {REPO_ROOT}')
print(f'HOST_WORKSPACE : {HOST_WORKSPACE}')
print(f'CONTAINER_PROJ : {CONTAINER_PROJ}')
print(f'CONTAINER_NAME : {CONTAINER_NAME}')

## Step 1 — Pre-flight checks

The agent will not start until these are green. Each check is cheap (no LLM calls).

In [ ]:
def check_container_running(name):
    out = subprocess.run(
        ['docker', 'ps', '--filter', f'name={name}', '--format', '{{.Names}}'],
        capture_output=True, text=True, timeout=10,
    )
    return name in out.stdout.split()


def check_command(cmd):
    return shutil.which(cmd)


def check_eda_agents():
    try:
        import eda_agents  # noqa: F401
        return True
    except ImportError:
        return False


def check_mcp_registered(repo_root):
    return (repo_root / '.mcp.json').is_file() or (repo_root / '.opencode.json').is_file()


if RUN_PREFLIGHT:
    container_ok = check_container_running(CONTAINER_NAME)
    claude_path  = check_command('claude')
    opencode_path = check_command('opencode')
    eda_ok = check_eda_agents()
    mcp_ok = check_mcp_registered(REPO_ROOT)

    print(f'  gf180 container running    : {container_ok}')
    print(f'  claude CLI on PATH         : {claude_path or "-- not found"}')
    print(f'  opencode CLI on PATH       : {opencode_path or "-- not found"}')
    print(f'  eda_agents importable      : {eda_ok}')
    print(f'  MCP registered at repo root: {mcp_ok}')
    print()

    if not container_ok:
        print('FIX: run `scripts/bootstrap_container.sh` from the repo root.')
    if not (claude_path or opencode_path):
        print('FIX: install Claude Code or opencode (see docs/eda_agents_setup.md step 3).')
    if not eda_ok:
        print('FIX: pip install -e <eda-agents path> in your active venv (see docs/eda_agents_setup.md step 2).')
    if not mcp_ok:
        print('FIX: run `eda-init` from the repo root (see docs/eda_agents_setup.md step 4).')
else:
    print('(skipped -- flip RUN_PREFLIGHT=True)')

## Step 2 — Stage the workspace

Copy `rtl/`, `tb/`, `librelane/` into `~/eda/designs/eda_agents_counter_tui/` so the container sees them at `/foss/designs/eda_agents_counter_tui/`.

In [ ]:
if RUN_STAGE:
    HOST_WORKSPACE.mkdir(parents=True, exist_ok=True)
    for sub in ('rtl', 'tb', 'librelane'):
        src = NB_DIR / sub
        dst = HOST_WORKSPACE / sub
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'  staged {src.relative_to(NB_DIR.parents[1])} -> {dst}')
    print()
    print(f'Workspace ready at: {HOST_WORKSPACE}')
    print(f'Container path    : {CONTAINER_PROJ}')
else:
    print('(skipped -- flip RUN_STAGE=True)')

## Step 3 — (Optional) cocotb sanity check before involving the agent

Quick ~10-second RTL sim. Prove the TB works locally so when the agent later runs the same `make test-counter`, you know it's the agent's setup that's at fault if it fails — not the testbench.

In [ ]:
if RUN_COCOTB_SANITY:
    cmd = [
        'docker', 'exec', CONTAINER_NAME, 'bash', '-lc',
        f'cd {CONTAINER_PROJ}/tb && make clean && make test-counter',
    ]
    print('$ ' + ' '.join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    print(proc.stdout[-2000:])
    if proc.returncode != 0:
        print('STDERR (tail):')
        print(proc.stderr[-1000:])
    print(f'returncode={proc.returncode}')
else:
    print('(skipped -- flip RUN_COCOTB_SANITY=True for a ~10s sanity check)')

## Step 4 — Invoke the `gf180-docker-digital` agent

Two paths, **pick one** depending on which CLI you installed. Both invoke the same agent and use the same MCP server (the `eda-agents` package), so the agent behaviour is identical.

Open a **new terminal** in the repo root, then paste the command. The agent will start a chat session — talk to it as you would a colleague.

### Recommended opening prompt

Whichever CLI you use, paste this as your first message after the agent loads:

> *Please harden the design at `/foss/designs/eda_agents_counter_tui/` end-to-end. The RTL is `rtl/counter.v`, the LibreLane config is `librelane/config.yaml`, the cocotb tests are in `tb/`. Run `make test-counter` first, then `make librelane`, then surface the signoff metrics from `final/metrics.csv`. Confirm the bind-mount path with me before launching anything.*

The agent will then walk through container check → cocotb → LibreLane → metrics, asking for approval at each big step. See [`docs/agent_walkthrough.md`](docs/agent_walkthrough.md) for what to expect at each turn.

In [ ]:
if PRINT_INVOCATIONS:
    print('=' * 72)
    print('Path A — Claude Code (Anthropic subscription)')
    print('=' * 72)
    print(textwrap.dedent(f'''
        cd {REPO_ROOT}
        claude --agent gf180-docker-digital
    ''').strip())
    print()
    print('=' * 72)
    print('Path B — opencode (multi-provider, pay-as-you-go)')
    print('=' * 72)
    print(textwrap.dedent(f'''
        cd {REPO_ROOT}
        opencode --agent gf180-docker-digital
    ''').strip())
    print()
    print('Inside the chat session, paste the prompt from the markdown cell above.')
    print('Expected wall time: 10-15 minutes (cocotb + LibreLane + a few agent turns).')
else:
    print('(skipped -- flip PRINT_INVOCATIONS=True)')

## Step 5 — After the agent finishes, verify the metrics

Once the agent has reported success in the chat, come back here, flip `RUN_METRICS_CHECK=True`, and run this cell. It parses `final/metrics.csv` from the host side (the bind-mount makes the container's output visible at `~/eda/designs/eda_agents_counter_tui/runs/<tag>/final/`) and asserts the four signoff numbers are zero.

In [ ]:
import csv

SIGNOFF_KEYS = [
    'design__instance__count__stdcell',
    'timing__setup_vio__count',
    'timing__hold_vio__count',
    'magic__drc_error__count',
    'klayout__drc_error__count',
    'design__lvs_error__count',
    'route__drc_errors',
    'power__total',
]

if RUN_METRICS_CHECK:
    # LibreLane writes runs/ next to config.yaml, which here is
    # HOST_WORKSPACE/librelane/.
    runs_dir = HOST_WORKSPACE / 'librelane' / 'runs'
    if not runs_dir.exists():
        print(f'No runs/ directory at {runs_dir}.')
        print('Did the agent finish LibreLane successfully? Check the chat transcript.')
    else:
        latest_run = sorted(runs_dir.iterdir(), key=lambda p: p.stat().st_mtime)[-1]
        metrics_path = latest_run / 'final' / 'metrics.csv'
        if not metrics_path.exists():
            print(f'metrics.csv missing under {latest_run}.')
        else:
            print(f'Reading {metrics_path}\n')
            found = {}
            with metrics_path.open() as fh:
                for row in csv.reader(fh):
                    if row and row[0] in SIGNOFF_KEYS:
                        found[row[0]] = row[1] if len(row) > 1 else ''
            for key in SIGNOFF_KEYS:
                print(f'  {key:45s} {found.get(key, "(missing)")}')
            print()
            zero_keys = [
                'timing__setup_vio__count',
                'timing__hold_vio__count',
                'magic__drc_error__count',
                'design__lvs_error__count',
                'route__drc_errors',
            ]
            offenders = [k for k in zero_keys if found.get(k, '0') not in ('0', '')]
            if offenders:
                print(f'FAIL: non-zero signoff metric(s): {offenders}')
            else:
                print('PASS: all hard-zero signoff metrics are 0.')
else:
    print('(skipped -- flip RUN_METRICS_CHECK=True after the agent reports success)')


## What you learned

- The `gf180-docker-digital` agent (shipped by `eda-agents`) loads the `flow.rtl2gds_gf180_docker` skill via MCP and uses it as a step-by-step playbook for the GF180MCU + LibreLane container flow.
- You drive the agent in a chat session; it asks for approval before any expensive step (container start, LibreLane run).
- The notebook is the **scaffold** (path guard, staging, metrics check); the agent is the **driver**. They share the same workspace via the bind-mount.
- The same agent is available under Claude Code and opencode with no behaviour change because both load the same `.md` definition and the same MCP server.

## Next

Tutorial 02 ([`../02_counter_python_api/`](../02_counter_python_api/)) shows how to do the same thing programmatically — no chat session, just ~30 lines of Python.